In [1]:
# Install PyMuPDF (if not already installed)
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 22.4 MB/s eta 0:00:00


In [2]:
# Upload the PDF
from google.colab import files
uploaded = files.upload()

Saving sample-sdf-document (1).pdf to sample-sdf-document (1).pdf


In [13]:
import fitz  # PyMuPDF
import re
import json
import pandas as pd
from datetime import datetime

In [14]:
doc = fitz.open(list(uploaded.keys())[0])

print("Number of pages:", len(doc))

Number of pages: 3


In [6]:
list(uploaded.keys())[0]

'sample-sdf-document (1).pdf'

In [15]:
def merge_bboxes(bboxes):
    """
    Merge multiple bounding boxes into one bounding box.
    Each bbox should be [x0, y0, x1, y1].
    """
    if not bboxes:
        return None

    x0 = min(b[0] for b in bboxes)
    y0 = min(b[1] for b in bboxes)
    x1 = max(b[2] for b in bboxes)
    y1 = max(b[3] for b in bboxes)

    return [x0, y0, x1, y1]


def normalize_date(date_text):
    """
    Converts dates like 20240315 to 2024-03-15.
    Leaves other date formats unchanged.
    """
    if re.fullmatch(r"\d{8}", date_text):
        return datetime.strptime(date_text, "%Y%m%d").strftime("%Y-%m-%d")

    return date_text


def extract_lines_with_bboxes(page):
    """
    Extracts text lines with bounding boxes using PyMuPDF word extraction.
    """
    words = page.get_text("words")

    lines = {}

    for word in words:
        x0, y0, x1, y1, text, block_no, line_no, word_no = word
        key = (block_no, line_no)

        if key not in lines:
            lines[key] = []

        lines[key].append({
            "text": text,
            "bbox": [x0, y0, x1, y1],
            "word_no": word_no
        })

    line_records = []

    for key, line_words in lines.items():
        line_words = sorted(line_words, key=lambda w: w["word_no"])

        line_text = " ".join(w["text"] for w in line_words)
        line_bbox = merge_bboxes([w["bbox"] for w in line_words])

        line_records.append({
            "text": line_text,
            "bbox": line_bbox,
            "block_line": key
        })

    return line_records

In [16]:
field_patterns = {
    "Vendor Name": r"\bCytiva\b",
    "Document Type": r"Certificate of Quality|Flow Kit Storage Conditions",
    "Document Date": r"\b\d{1,2}\s+[A-Za-z]+,\s+\d{4}\b",
    "Product": r"Product:\s*(.+)",
    "Lot Number": r"Lot Number:\s*([A-Za-z0-9]+)",
    "Product Article Number": r"Product Article Number:\s*([\d\s]+)",
    "Date of Manufacture": r"Date of Manufacture:\s*(\d{8})",
    "Expiration Date": r"Expiration Date:\s*(\d{8})",
    "Product Description": r"Product Description:\s*(.+)",
    "Revision Number": r"Rev\s+[A-Z]+",
    "Country of Origin": r"Country of Origin:\s*([A-Za-z]+)"
}

extracted_fields = []

for page_number, page in enumerate(doc, start=1):
    lines = extract_lines_with_bboxes(page)

    for line in lines:
        line_text = line["text"]

        for label, pattern in field_patterns.items():
            match = re.search(pattern, line_text)

            if match:
                extracted_text = match.group(0)

                value = match.group(1) if match.groups() else extracted_text

                if label in ["Date of Manufacture", "Expiration Date"]:
                    normalized_value = normalize_date(value)
                else:
                    normalized_value = value

                extracted_fields.append({
                    "page": page_number,
                    "label": label,
                    "text": extracted_text,
                    "value": value,
                    "normalized_value": normalized_value,
                    "bbox": line["bbox"]
                })

print("Extracted Fields:")
for field in extracted_fields:
    print(field)

Extracted Fields:
{'page': 1, 'label': 'Vendor Name', 'text': 'Cytiva', 'value': 'Cytiva', 'normalized_value': 'Cytiva', 'bbox': [547.4367065429688, 87.58555603027344, 576.0117797851562, 98.43190002441406]}
{'page': 1, 'label': 'Document Date', 'text': '3 June, 2022', 'value': '3 June, 2022', 'normalized_value': '3 June, 2022', 'bbox': [35.998695373535156, 149.2158203125, 94.16860961914062, 160.7857666015625]}
{'page': 1, 'label': 'Document Type', 'text': 'Flow Kit Storage Conditions', 'value': 'Flow Kit Storage Conditions', 'normalized_value': 'Flow Kit Storage Conditions', 'bbox': [35.998695373535156, 187.12989807128906, 259.8073425292969, 199.69619750976562]}
{'page': 1, 'label': 'Vendor Name', 'text': 'Cytiva', 'value': 'Cytiva', 'normalized_value': 'Cytiva', 'bbox': [35.998695373535156, 556.494873046875, 126.14053344726562, 568.0648193359375]}
{'page': 2, 'label': 'Document Type', 'text': 'Certificate of Quality', 'value': 'Certificate of Quality', 'normalized_value': 'Certificate

In [17]:
table_rows = []

for page_number, page in enumerate(doc, start=1):
    lines = extract_lines_with_bboxes(page)

    for line in lines:
        text = line["text"].strip()

        # Page 1 storage conditions table
        storage_match = re.match(
            r"(.+?)\s+(\d{8})\s+(.+C.*)",
            text
        )

        if storage_match:
            table_rows.append({
                "page": page_number,
                "table_type": "Storage Conditions",
                "description": storage_match.group(1),
                "part_number": storage_match.group(2),
                "operating_temperature": storage_match.group(3),
                "bbox": line["bbox"]
            })

        # Page 3 product release criteria table
        release_patterns = [
            r"^(Autoclave\s*-\s*Pump tubing)\s+(121C\s*>\s*15 min)\s+(Conforms)$",
            r"^(Gamma Irradiation\s*-\s*Inlets)\s+(25\.0\s*-\s*40\.0 kGy)\s+(Conforms)$",
            r"^(Flow Rate Test)\s+(Per SOP-QC-\d+)\s+(Pass)$",
            r"^(Pressure Integrity)\s+(Max .+?hold)\s+(Pass)$",
            r"^(Visual Inspection)\s+(No defects visible)\s+(Pass)$",
            r"^(Package Integrity)\s+(Sealed, no damage)\s+(Pass)$"
        ]

        for pattern in release_patterns:
            match = re.match(pattern, text)

            if match:
                table_rows.append({
                    "page": page_number,
                    "table_type": "Product Release Criteria",
                    "description": match.group(1),
                    "test_method": match.group(2),
                    "result": match.group(3),
                    "bbox": line["bbox"]
                })

print("\nExtracted Table Rows:")
for row in table_rows:
    print(row)


Extracted Table Rows:
{'page': 1, 'table_type': 'Storage Conditions', 'description': 'Instructions', 'part_number': '28960345', 'operating_temperature': 'and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits', 'bbox': [35.998695373535156, 266.095703125, 564.1392211914062, 277.6656494140625]}


In [18]:
fields_df = pd.DataFrame(extracted_fields)
tables_df = pd.DataFrame(table_rows)

print("Fields DataFrame:")
print(fields_df)

print("\nTables DataFrame:")
print(tables_df)

Fields DataFrame:
    page                   label  \
0      1             Vendor Name   
1      1           Document Date   
2      1           Document Type   
3      1             Vendor Name   
4      2           Document Type   
5      2             Vendor Name   
6      2             Vendor Name   
7      2             Vendor Name   
8      2         Revision Number   
9      2                 Product   
10     2              Lot Number   
11     2  Product Article Number   
12     2     Date of Manufacture   
13     2     Product Description   
14     2         Expiration Date   
15     2       Country of Origin   
16     3             Vendor Name   
17     3           Document Type   
18     3             Vendor Name   
19     3             Vendor Name   
20     3             Vendor Name   
21     3         Revision Number   
22     3       Country of Origin   

                                                 text  \
0                                              Cytiva   
1  

In [19]:
output = {
    "extracted_fields": extracted_fields,
    "table_rows": table_rows
}

with open("sdf_extraction_output.json", "w", encoding="utf-8") as file:
    json.dump(output, file, indent=4, ensure_ascii=False)

print("Extraction results saved to sdf_extraction_output.json")

Extraction results saved to sdf_extraction_output.json


In [21]:
annotated_doc = fitz.open(list(uploaded.keys())[0])

for field in extracted_fields:
    page_index = field["page"] - 1
    page = annotated_doc[page_index]

    if field["bbox"]:
        rect = fitz.Rect(field["bbox"])
        page.draw_rect(rect, color=(1, 0, 0), width=1.5)
        page.insert_text(
            (rect.x0, rect.y0 - 5),
            field["label"],
            fontsize=7,
            color=(1, 0, 0)
        )

for row in table_rows:
    page_index = row["page"] - 1
    page = annotated_doc[page_index]

    if row["bbox"]:
        rect = fitz.Rect(row["bbox"])
        page.draw_rect(rect, color=(0, 0, 1), width=1.2)

annotated_doc.save("sdf_annotated_output.pdf")
annotated_doc.close()

print("Annotated PDF saved as sdf_annotated_output.pdf")

Annotated PDF saved as sdf_annotated_output.pdf


In [23]:
doc = fitz.open(list(uploaded.keys())[0])

all_tables = []

for page_number, page in enumerate(doc, start=1):
    tables = page.find_tables()

    print(f"\nPage {page_number}: {len(tables.tables)} table(s) found")

    for table_index, table in enumerate(tables.tables, start=1):
        extracted_table = table.extract()

        print(f"\nTable {table_index} on Page {page_number}:")
        for row in extracted_table:
            print(row)

        all_tables.append({
            "page": page_number,
            "table_index": table_index,
            "bbox": list(table.bbox),
            "data": extracted_table
        })

Consider using the pymupdf_layout package for a greatly improved page layout analysis.

Page 1: 0 table(s) found

Page 2: 0 table(s) found

Page 3: 1 table(s) found

Table 1 on Page 3:
['Description', 'Test Method', 'Result']
['Autoclave - Pump tubing', '121C > 15 min', 'Conforms']
['Gamma Irradiation - Inlets', '25.0 - 40.0 kGy', 'Conforms']
['Flow Rate Test', 'Per SOP-QC-042', 'Pass']
['Pressure Integrity', 'Max 0.5 bar, 5 min hold', 'Pass']
['Visual Inspection', 'No defects visible', 'Pass']
['Package Integrity', 'Sealed, no damage', 'Pass']


In [24]:
for table_info in all_tables:
    df = pd.DataFrame(table_info["data"])

    print(f"\nPage {table_info['page']} - Table {table_info['table_index']}")
    print(df)


Page 3 - Table 1
                            0                        1         2
0                 Description              Test Method    Result
1     Autoclave - Pump tubing            121C > 15 min  Conforms
2  Gamma Irradiation - Inlets          25.0 - 40.0 kGy  Conforms
3              Flow Rate Test           Per SOP-QC-042      Pass
4          Pressure Integrity  Max 0.5 bar, 5 min hold      Pass
5           Visual Inspection       No defects visible      Pass
6           Package Integrity        Sealed, no damage      Pass
